# インボイス確認ツール(Colab 版)

[GitHub Pages 版](https://masakiniwa.github.io/Invoice/)と同じ画面を、Google Colab の計算資源(GPU)で動かします。

- 文字認識(PaddleOCR)を Colab の **GPU** で実行します。ブラウザにモデルをダウンロードしないので、スマホでも軽く動きます
- 起動すると **公開リンク**(Cloudflare)と、毎回変わる **パスワード** が表示されます。パスワードを知っている人だけが使えます
- 読み込んだ画像は Colab のメモリ上で処理し、保存しません(履歴・設定はこれまでどおり使う人のブラウザに保存されます)

## 使い方
1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選ぶ(CPU でも動きます)
2. 下のセルを上から順に実行する(▶ ボタン)
3. 表示された「アプリを開く」からログインする

> 公開リンクは、このノートブックを止めるかランタイムが切断されると使えなくなります。パスワード入りのリンクは人に送らないでください。

In [ ]:
#@title 設定 { display-mode: "form" }
#@markdown **UI の取得方法**: `pages` = 公開中の版をダウンロード(速い)/ `build` = 下のブランチのソースからビルド(開発中の版を試すとき、数分かかります)
SOURCE = "pages"  #@param ["pages", "build"]
#@markdown リポジトリのブランチ(サーバーのプログラムもこのブランチのものを使います)
BRANCH = "main"  #@param {type:"string"}
#@markdown Cloudflare の公開リンクを作る(オフにすると Colab 内のリンクのみ。このアカウントのブラウザでだけ開けます)
USE_TUNNEL = True  #@param {type:"boolean"}
REPO = "https://github.com/MasakiNiwa/Invoice.git"

In [ ]:
#@title 1. 準備(初回 1〜2 分) { display-mode: "form" }
import os, subprocess, sys
REPO_DIR = "/content/Invoice"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "colab"))
import importlib, launcher
importlib.reload(launcher)
print("GPU:", "あり" if launcher.has_gpu() else "なし(CPU で動かします)")
launcher.setup(source=SOURCE)

In [ ]:
#@title 2. 起動(リンクとパスワードを表示) { display-mode: "form" }
info = launcher.start(tunnel=USE_TUNNEL)

### 困ったとき
- 画面が開かない: 下のセルでログを確認してください。「2. 起動」をもう一度実行すると、新しいリンクとパスワードで起動し直します
- 公開リンクを作れない: Cloudflare の一時的な不調のことがあります。少し待ってから「2. 起動」を再実行するか、Colab 内のリンクを使ってください

In [ ]:
#@title ログを見る { display-mode: "form" }
launcher.logs("server")
launcher.logs("cloudflared")

In [ ]:
#@title 停止 { display-mode: "form" }
launcher.stop()